# Experiment 6e - ephemeral n-think in complex games
Experiment 6 tested r4/r10-think only in the non-ephemeral version (`reads_own_reasoning=True`), based on the conclusion of experiment 5.
However, experiments 3 and 5 passed `reads_own_reasoning=ephemeral`, so the results labelled as ephemeral are actually the non-ephemeral ones and vice versa. With the labels corrected, experiment 5 suggests that ephemeral r4/r10 score higher than the non-ephemeral versions, but only on Simple and on level 1 of Cooking.

Here we test whether this holds in the complex games: we run the missing ephemeral er4/er10-think (`reads_own_reasoning=False`) on the same games, levels and seeds of experiment 6, and compare them with the existing 0-, e1-, r4- and r10-think results.

# Setup
On Google Colab this clones the repository, installs TextWorld and moves into `Code/Experiments`. Locally (e.g. in the Docker container) it does nothing.

In [1]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Self-evaluation/self_evaluation.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone
        # sparse clone: only the folders needed by the experiments are downloaded (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        # already cloned in this runtime: only fetch the new commits (fails safely instead of overwriting local results)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    # the clone may come from another experiment's notebook: add the folders this one needs
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add",
                    "Code/Self-evaluation", "Code/Games/coin_collector", "Code/Games/cooking",
                    "Code/Games/treasure_hunter", "Code/Experiments/Experiment 6"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "textworld"], check=True)
    os.chdir(f"{repo_dir}/Code/Experiments")

print("Working directory:", os.getcwd())

Working directory: /content/LLMs-Play-Textual-Games/Code/Experiments


On Colab the results are saved on Google Drive, because the files of the Colab machine are deleted together with it.
If mounting fails, use the VS Code command *Colab: Mount Google Drive to Server...*

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [2]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. T4)"
! nvidia-smi

Tue Oct  6 19:48:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import pickle
sys.path.append("../Self-evaluation")

from self_evaluation import play, LLMAgentSelfEvaluate

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

# Notebook variables

In [4]:
exp_number = 6
games_folder = "../Games"
seeds = range(1,21) # same seeds of the r4/r10-think results of experiment 6
n_episodes = 1
max_steps = 100
n_think_indices = [4,10]

levels = {"coin_collector": 14, "cooking": 9, "treasure_hunter": 27}
run_levels = {"cooking": range(1, 10)} # levels to play: extend it with the other games once these are done

# episodes stuck in a loop (at most loop_max_distinct different turns among the last loop_window) are stopped early
loop_window = 20
loop_max_distinct = 2

# new results and logs: on Colab they go to Google Drive, so they survive the deletion of the Colab machine
if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results_folder = f"/content/drive/MyDrive/LLMs-Play-Textual-Games/Experiment {exp_number}"
else:
    results_folder = f"./Experiment {exp_number}"
print("Results folder:", results_folder)

with open(f"{games_folder}/max_scores_cooking.pickle", "rb") as f:
    max_scores_cooking = pickle.load(f)
for game in levels.keys():
    os.makedirs(f"{results_folder}/logs/{game}", exist_ok=True)

# Game running
The results are saved after every seed: if the run is interrupted, running this cell again resumes from the first seed not yet played.

In [ ]:
for game, game_levels in run_levels.items():
    for level in game_levels:
        for n in n_think_indices:
            results_path = f"{results_folder}/er{n}think_{game}_level{level}.pickle"
            results_all_seeds = []
            if os.path.exists(results_path):
                with open(results_path, "rb") as f:
                    results_all_seeds = pickle.load(f)
            for seed in seeds[len(results_all_seeds):]: # skip the seeds already played
                results = play(LLMAgentSelfEvaluate(selfeval_turns=n,
                                                    reads_own_reasoning=False, # ephemeral
                                                    random_selfeval=True,
                                                    show_speed=True, # tokens per minute, updated at every turn
                                                    log=f"{results_folder}/logs/{game}/er{n}think_level{level}-seed{seed}.log"
                                                    ),
                               f"{games_folder}/{game}/level{level}-seed{seed}.z8",
                               max_steps=max_steps,
                               n_episodes=n_episodes,
                               loop_window=loop_window,
                               loop_max_distinct=loop_max_distinct)
                results_all_seeds.append(results[0])
                with open(results_path, "wb") as f:
                    pickle.dump(results_all_seeds, f)

../Games/cooking/level1-seed1.z8 .  	avg. steps:  40.0; avg. score:  5.0 / 5; total execution time: 740 s.
../Games/cooking/level1-seed2.z8 

# Comparison
Average normalized final score per level, with the number of seeds in brackets (e1-think has only 10 seeds).
Execution times are not compared here: the episodes stopped because of a loop are shorter than in the other runs.

In [ ]:
import numpy as np

models = ["0", "e1", "r4", "er4", "r10", "er10"]
for game, game_levels in run_levels.items():
    print(game)
    print("level" + "".join(f"{model + '-think':>14}" for model in models))
    for level in game_levels:
        row = []
        for model in models:
            folder = results_folder if model.startswith("er") else f"./Experiment {exp_number}" # new results vs results of experiment 6
            results_path = f"{folder}/{model}think_{game}_level{level}.pickle"
            if not os.path.exists(results_path):
                row.append("-")
                continue
            with open(results_path, "rb") as f:
                results_all_seeds = pickle.load(f)[:len(seeds)]
            final_scores = np.array([run[-1][1] for run in results_all_seeds], dtype=float)
            if game == "cooking":
                final_scores /= max_scores_cooking[level - 1]
            row.append(f"{np.mean(final_scores):.2f} ({len(final_scores)})")
        print(f"{level:>5}" + "".join(f"{cell:>14}" for cell in row))